# Segment correlation distance experiment

This notebook computes the staff-approved Pearson-distance transform

$D_{ij} = \sqrt{2 \times (1 - r_{ij})}$

The expected repo artifact is `data/docs/segment_monthly_series.parquet`, but this checkout does not contain it. The real source in this session is the downloaded parquet under the user profile, so the notebook checks both paths and uses the real data that exists.

In [ ]:
from pathlib import Path
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "backend").exists() and (candidate / "notebooks").exists():
        repo_root = candidate
        break
repo_path = repo_root / "data" / "docs" / "segment_monthly_series.parquet"
download_path = Path.home() / "Downloads" / "segment_monthly_series.parquet"
series_path = next((p for p in [repo_path, download_path] if p.exists()), None)
print("repo_path_exists", repo_path.exists())
print("download_path_exists", download_path.exists())
print("repo_root", repo_root)
print("series_path", series_path)
if series_path is None:
    raise FileNotFoundError("No segment_monthly_series.parquet found in repo or Downloads")


In [ ]:
import pandas as pd
import numpy as np
series = pd.read_parquet(series_path)
print(series.head().to_string(index=False))
print("shape", series.shape)
print("columns", list(series.columns))
print("states", sorted(series["property_state"].unique().tolist()))
print("months", series["monthly_reporting_period"].nunique())
print("min_month", series["monthly_reporting_period"].min())
print("max_month", series["monthly_reporting_period"].max())


In [ ]:
wide = series.pivot_table(index="monthly_reporting_period", columns="property_state", values="delinquency_rate", aggfunc="mean").sort_index()
corr = wide.corr(method="pearson")
corr_array = corr.to_numpy(copy=True)
distance_array = np.sqrt(np.clip(2 * (1 - corr_array), 0, None))
distance_df = pd.DataFrame(distance_array, index=corr.index, columns=corr.columns)
for idx in range(len(distance_df.columns)):
    distance_df.iat[idx, idx] = 0.0
print("corr_shape", corr.shape)
print("distance_shape", distance_df.shape)
display(corr.head())
display(distance_df.head())


## Validate the matrix properties

Check symmetry, diagonal agreement, and bounds.

In [ ]:
cutoff = np.sqrt(2 * (1 - 0.95))
validation = {
    "symmetry_ok": bool(np.allclose(corr.to_numpy(), corr.to_numpy().T, atol=1e-12)),
    "diag_corr_ok": bool(np.allclose(np.diag(corr.to_numpy()), 1.0, atol=1e-12)),
    "diag_distance_ok": bool(np.allclose(np.diag(distance_df.to_numpy()), 0.0, atol=1e-12)),
    "distance_nonnegative": bool((distance_df.to_numpy()[~np.eye(distance_df.shape[0], dtype=bool)] >= 0).all()),
    "pearson_offdiagonal_min": float(corr.to_numpy()[~np.eye(corr.shape[0], dtype=bool)].min()),
    "pearson_offdiagonal_max": float(corr.to_numpy()[~np.eye(corr.shape[0], dtype=bool)].max()),
    "distance_offdiagonal_min": float(distance_df.to_numpy()[~np.eye(distance_df.shape[0], dtype=bool)].min()),
    "distance_offdiagonal_max": float(distance_df.to_numpy()[~np.eye(distance_df.shape[0], dtype=bool)].max()),
    "cutoff_d": float(cutoff),
}
print(validation)


## Pairwise examples and threshold equivalence

The threshold `r >= 0.95` is equivalent to `D <= sqrt(2 * (1 - 0.95))`.

In [ ]:
rows = []
for i, state_i in enumerate(corr.columns):
    for state_j in corr.columns[i + 1:]:
        r = float(corr.loc[state_i, state_j])
        d = float(np.sqrt(2 * (1 - r)))
        rows.append({
            "state_i": state_i,
            "state_j": state_j,
            "pearson_r": r,
            "distance_d": d,
            "r_ge_0_95": r >= 0.95,
            "d_le_cutoff": d <= cutoff,
        })
pair_df = pd.DataFrame(rows)
display(pair_df.sort_values("pearson_r", ascending=False).head(5))
print("threshold_match", bool(pair_df["r_ge_0_95"].equals(pair_df["d_le_cutoff"])))


## Compare to the existing graph export

The original graph keeps all state pairs with `r >= 0.95`; the distance cutoff should select the same pairs.

In [ ]:
import json
graph_path = repo_root / "backend" / "risk-engine" / "src" / "main" / "resources" / "segment_correlation_graph.json"
with graph_path.open("r", encoding="utf-8") as f:
    graph = json.load(f)
existing_pairs = {(min(e["source"], e["target"]), max(e["source"], e["target"])) for e in graph["edges"]}
threshold_pairs = {(min(r.state_i, r.state_j), max(r.state_i, r.state_j)) for _, r in pair_df.iterrows() if r["r_ge_0_95"]}
print("graph_edge_count", len(graph["edges"]))
print("threshold_edge_count", len(threshold_pairs))
print("graph_matches_threshold", existing_pairs == threshold_pairs)
